# Wave Glider + TAO OSSE: North-shifted 3-cell diamonds

Each config in `configs/with_TAO/north_shift/` pairs 6 gliders (2 lon columns, 3 lat rows
offset +0.5 deg N of TAO (2S excluded)) with the 140W TAO moorings. Every glider row sits exactly
between two TAO moorings, forming 3 overlapping 4-point diamonds (2 TAO N/S +
2 glider E/W). Each diamond gives its own independent plane-fit w estimate.

**To rerun for a different pattern**, point `CONFIG_FILES`/`OUTDIR`/`TITLE` at
the new configs — everything else is generic over however many cells a config
defines. **To rerun for a different depth range** (e.g. an 8 m glider that
can't see the near-surface), just change `MIN_DEPTH`/`MAX_DEPTH` below —
`compute_w_planefit` assumes w=0 at `MIN_DEPTH`, not necessarily the surface.

In [ ]:
import json
import os
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

os.chdir('/home/edavenport/analysis/tpose24-osse')  # pin working directory
sys.path.insert(0, '/home/edavenport/analysis/tpose24-osse')

from osse_tools import (load_model, load_cells, sample_fields,
                        compute_w_planefit, sample_model_w,
                        plot_w_comparison, plot_velocity_map)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Configuration

In [ ]:
RUN_DIR   = '/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons'
ITERS     = list(range(36, 26173, 36))

MIN_DEPTH = 0     # shallowest depth sampled (m); w=0 is assumed here, not necessarily the surface
MAX_DEPTH = 120
DZ_OBS    = 2

TITLE     = 'North-shifted 3-cell diamonds'
OUTDIR    = f'north_shift_3cell/{int(MIN_DEPTH)}to{int(MAX_DEPTH)}m'

CONFIG_FILES = [
    'configs/with_TAO/north_shift/w0.25.json',
    'configs/with_TAO/north_shift/w0.5.json',
    'configs/with_TAO/north_shift/w0.75.json',
    'configs/with_TAO/north_shift/w1.0.json',
]

## Load model (once)

In [ ]:
ds = load_model(RUN_DIR, ITERS)
ds = ds.sel(time=slice('2012-10-11', None))  # exclude spin-up

## Sample every position once

Cells across configs and widths reuse the same TAO moorings, so sample the
union of all unique positions in a single batched call instead of resampling
shared points once per cell.

In [ ]:
configs = {Path(f).stem: load_cells(f) for f in CONFIG_FILES}  # {width_key: [(center_lat, positions), ...]}
width_vals = {wk: float(wk[1:]) for wk in configs}             # 'w0.5' -> 0.5

all_positions = sorted({p for cells in configs.values() for _, pos in cells for p in pos})
uv_all = sample_fields(ds, all_positions, vars=('UVEL', 'VVEL'),
                       max_depth=MAX_DEPTH, dz_obs=DZ_OBS, min_depth=MIN_DEPTH).compute()
pos_idx = {p: i for i, p in enumerate(all_positions)}

## Run all cells

Plane fits reuse the already-computed `uv_all`, so only `sample_model_w`
(a fresh hull read per cell) is expensive — run those concurrently.

In [ ]:
def run_cell(width_key, center_lat, positions):
    uv = uv_all.isel(glider=[pos_idx[p] for p in positions])
    w_est = compute_w_planefit(uv)['w_est']
    w_model = sample_model_w(ds, positions, max_depth=MAX_DEPTH, dz_obs=DZ_OBS,
                             min_depth=MIN_DEPTH, spatial_mean=True)
    bias = w_est - w_model
    return width_key, center_lat, dict(
        positions=positions, w_est=w_est, w_model=w_model, bias=bias,
        rms=float(np.sqrt((bias ** 2).mean())), mean_bias=float(bias.mean()),
    )

jobs = [(wk, cl, pos) for wk, cells in configs.items() for cl, pos in cells]
with ThreadPoolExecutor(max_workers=8) as ex:
    out = list(ex.map(lambda j: run_cell(*j), jobs))

results = {}
for wk, cl, r in out:
    results.setdefault(wk, {})[cl] = r
    print(f"{wk} cell={cl:+.1f}  RMS={r['rms']:.3e}  bias={r['mean_bias']:+.3e}")

## Per-cell figures

In [ ]:
for wk, cells in results.items():
    for cl, r in cells.items():
        outdir = os.path.join(OUTDIR, wk, f'cell_{cl:g}')
        os.makedirs(outdir, exist_ok=True)
        fig = plot_w_comparison(r['w_est'], r['w_model'], point_depth=-50)
        fig.suptitle(f'{TITLE}: {wk}, cell {cl:+.1f}\u00b0N', fontsize=13, y=1.01)
        plt.savefig(f'{outdir}/w_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()

## Velocity maps (per width, cells highlighted)

One map per width covering the whole array (not per individual diamond, since
the 3 diamonds heavily overlap) — each cell gets its own color; TAO moorings
shared by two cells are drawn as a big circle (first cell) under a small star
(second cell) so both colors show.

In [ ]:
for wk, cell_list in configs.items():
    positions_w = sorted({p for _, pos in cell_list for p in pos})
    cells_plot = [(f'{cl:+.1f}', pos, f'C{i}') for i, (cl, pos) in enumerate(cell_list)]
    fig = plot_velocity_map(ds, positions_w, max_depth=MAX_DEPTH, cells=cells_plot)
    fig.suptitle(f'{TITLE}: {wk}  ({int(MIN_DEPTH)}\u2013{int(MAX_DEPTH)} m)', fontsize=12, y=1.02)
    outdir = os.path.join(OUTDIR, wk)
    os.makedirs(outdir, exist_ok=True)
    plt.savefig(f'{outdir}/velocity_map.png', dpi=150, bbox_inches='tight')
    plt.show()

## Summary comparison across width (per cell)

Mirrors the scale-sweep summary used for the other array shapes, but grouped
by diamond (cell) rather than by config family, since width is the axis of
interest here.

In [ ]:
by_cell = {}
for wk, cells in results.items():
    for cl, r in cells.items():
        by_cell.setdefault(cl, {})[wk] = r

for cl, per_width in sorted(by_cell.items()):
    wks = sorted(per_width, key=lambda k: width_vals[k])
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(wks)))

    # Reference "true" signal for this cell: average the (nearly identical) per-width model series
    model_mean_series = sum(per_width[wk]['w_model'] for wk in wks) / len(wks)
    Z0 = model_mean_series.depth.values
    model_prof = model_mean_series.mean('time').values
    model_std  = model_mean_series.std('time').values
    signal_mag = float(model_mean_series.std())  # typical size of the true w signal, for scale

    fig, axes = plt.subplots(2, 3, figsize=(22, 10))

    axes[0, 2].fill_betweenx(Z0, model_prof - model_std, model_prof + model_std,
                            color='0.3', alpha=0.15, label='model ±1σ (time)')
    axes[0, 2].plot(model_prof, Z0, color='0.3', lw=2, label='model mean')

    for wk, color in zip(wks, colors):
        r = per_width[wk]
        T, Z = r['bias'].time.values, r['bias'].depth.values
        axes[0, 0].plot(T, r['bias'].mean('depth').values, color=color, lw=1.2, label=wk)
        axes[0, 1].plot(r['bias'].mean('time').values, Z, color=color, lw=1.5, label=wk)
        axes[1, 0].plot(T, r['bias'].sel(depth=-50, method='nearest').values, color=color, lw=1.2, label=wk)
        axes[0, 2].plot(r['bias'].mean('time').values, Z, color=color, lw=1.5, label=f'{wk} bias')

    axes[0, 0].axhline(0, color='k', lw=0.5, ls=':')
    axes[0, 0].set_ylabel('Depth-mean bias (m s⁻¹)'); axes[0, 0].set_title('Depth-mean w error vs time')
    axes[0, 0].legend(title='Width', fontsize=9); axes[0, 0].grid(alpha=0.3)
    axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    axes[0, 0].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
    plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=30, ha='right')

    axes[0, 1].axvline(0, color='k', lw=0.7, ls=':')
    axes[0, 1].set_xlabel('Time-mean bias (m s⁻¹)'); axes[0, 1].set_ylabel('Depth (m)')
    axes[0, 1].set_title('Time-mean w error vs depth')
    axes[0, 1].legend(title='Width', fontsize=9); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].axhline(0, color='k', lw=0.5, ls=':')
    axes[1, 0].set_ylabel('Bias at 50 m (m s⁻¹)'); axes[1, 0].set_title('w error at 50 m vs time')
    axes[1, 0].legend(title='Width', fontsize=9); axes[1, 0].grid(alpha=0.3)
    axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    axes[1, 0].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
    plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=30, ha='right')

    xw = [width_vals[wk] for wk in wks]
    axes[1, 1].plot(xw, [per_width[wk]['rms'] for wk in wks], 'o-', color='C0', lw=1.5, label='RMS error')
    axes[1, 1].plot(xw, [abs(per_width[wk]['mean_bias']) for wk in wks], 's--', color='C1', lw=1.5, label='|mean bias|')
    axes[1, 1].set_xlabel('lon offset (deg)'); axes[1, 1].set_ylabel('m s⁻¹')
    axes[1, 1].set_title('Depth-and-time mean error vs width')
    axes[1, 1].legend(fontsize=9); axes[1, 1].grid(alpha=0.3)

    axes[0, 2].axvline(0, color='k', lw=0.7, ls=':')
    axes[0, 2].set_xlabel('w (m s⁻¹)'); axes[0, 2].set_ylabel('Depth (m)')
    axes[0, 2].set_title('Model signal (mean±σ) vs bias profiles')
    axes[0, 2].legend(fontsize=7, loc='best')
    axes[0, 2].grid(alpha=0.3)

    rel_rms  = [per_width[wk]['rms'] / signal_mag * 100 for wk in wks]
    rel_bias = [abs(per_width[wk]['mean_bias']) / signal_mag * 100 for wk in wks]
    axes[1, 2].plot(xw, rel_rms, 'o-', color='C0', lw=1.5, label='RMS / signal σ')
    axes[1, 2].plot(xw, rel_bias, 's--', color='C1', lw=1.5, label='|bias| / signal σ')
    axes[1, 2].axhline(100, color='k', lw=0.5, ls=':')
    axes[1, 2].set_xlabel('lon offset (deg)'); axes[1, 2].set_ylabel('% of signal σ')
    axes[1, 2].set_title('Relative error vs width')
    axes[1, 2].legend(fontsize=9); axes[1, 2].grid(alpha=0.3)

    fig.suptitle(f'{TITLE}: cell {cl:+.1f}°N  ({int(MIN_DEPTH)}–{int(MAX_DEPTH)} m)', fontsize=13)
    fig.tight_layout()
    cell_dir = os.path.join(OUTDIR, f'cell_{cl:g}')
    os.makedirs(cell_dir, exist_ok=True)
    plt.savefig(os.path.join(cell_dir, 'summary_error_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Summary statistics

In [ ]:
header = f"{'Cell':>8}  {'Width':>7}  {'RMS (m/s)':>12}  {'Mean bias':>14}  {'w_est std':>12}  {'w_model std':>12}"
print(header); print('-' * len(header))
for cl, per_width in sorted(by_cell.items()):
    for wk in sorted(per_width, key=lambda k: width_vals[k]):
        r = per_width[wk]
        print(f"{cl:>+8.1f}  {width_vals[wk]:>7.2f}  {r['rms']:>12.3e}  {r['mean_bias']:>+14.3e}  "
              f"{float(r['w_est'].std()):>12.3e}  {float(r['w_model'].std()):>12.3e}")